# Demo: varying interclonal cell mixing rate

Left/right tumor clusters with infiltration rate
$r \in \{0, 0.08, 0.16, 0.24, 0.32, 0.40\}$
(fraction of each tumor type placed in the opposite compartment).

## Requirements
`numpy`, `pandas`, `matplotlib`

## Paths (repository root)
| Role | Path |
|------|------|
| **Input** (optional) | `data/cell_anno.tsv` |
| **Output** | `output/vary_mixing_rate/<condition>/` |

### Input file
Same as `01_vary_shape.ipynb`: header-free `barcode<TAB>clone_label` TSV under `data/cell_anno.tsv`.

### Output files (per condition)
| File | Format |
|------|--------|
| `tissue_positions_list.csv` | no header; `barcode, in_tissue, x, y, pixel_row, pixel_col` |
| `spot_anno_pattern.tsv` | TSV with header; barcode index + `spot_anno` |
| `barcodes.tsv.gz` | one barcode per line |

Conditions: `mixing_0`, `0.08`, `0.16`, `0.24`, `0.32`, `0.40`.

## Run
```bash
jupyter nbconvert --to notebook --execute 03_vary_mixing_rate.ipynb
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

# Run this notebook from the repository root (directory that contains this file).
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "_spatial_pattern_utils.py").exists():
    raise FileNotFoundError(
        "Cannot find _spatial_pattern_utils.py in cwd. "
        "cd to the demo repository root before running."
    )
sys.path.insert(0, str(REPO_ROOT))

from _spatial_pattern_utils import (
    DEFAULT_CELL_ANNO,
    DEFAULT_OUTPUT_DIR,
    SEED,
    TARGET_PER_TYPE,
    load_or_make_barcodes_by_type,
    make_hex_grid,
    map_barcodes_to_labels,
    plot_spatial,
    resolve_repo_root,
    mixing_rate_schedule,
    pattern_vary_mixing_rate,
    write_pattern_dir,
)

REPO_ROOT = resolve_repo_root()
# Optional input: header-free TSV with columns barcode, clone_label
CELL_ANNO = REPO_ROOT / DEFAULT_CELL_ANNO

OUT_ROOT = REPO_ROOT / DEFAULT_OUTPUT_DIR / "vary_mixing_rate"

grid = make_hex_grid()
rates = mixing_rate_schedule(n=6, rate_max=0.4)
barcodes_by_type = load_or_make_barcodes_by_type(CELL_ANNO, seed=SEED)
print("REPO_ROOT:", REPO_ROOT)
print("Mixing rates:", [round(float(r), 2) for r in rates])
print("Clone sizes:", TARGET_PER_TYPE)
print("Input cell_anno:", CELL_ANNO if CELL_ANNO.exists() else "(missing → synthetic barcodes)")
print("Output root:", OUT_ROOT)


In [ ]:
def mixing_display_name(r: float, i: int) -> str:
    if i == 0 or abs(r) < 1e-12:
        return "mixing_0"
    return f"{r:.2f}"

pattern_dirs = {}
for i, r in enumerate(rates):
    labels = pattern_vary_mixing_rate(grid, float(r), seed=SEED + 2000 + i)
    display = mixing_display_name(float(r), i)
    barcodes = map_barcodes_to_labels(labels, barcodes_by_type, seed=SEED + 3000 + i)
    outdir = write_pattern_dir(OUT_ROOT / display, grid, labels, barcodes)
    pattern_dirs[display] = (outdir, labels, float(r))
    print(f"{display:10s}  r={r:.2f}  -> {outdir}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (display, (_, labels, r)) in zip(axes.ravel(), pattern_dirs.items()):
    plot_spatial(ax, grid, labels, title=f"{display} (r={r:.2f})")
axes[0, 0].legend(loc="upper left", fontsize=7, markerscale=2)
fig.suptitle("Varying interclonal cell mixing rate", fontsize=12)
fig.tight_layout()
OUT_ROOT.mkdir(parents=True, exist_ok=True)
fig_path = OUT_ROOT / "vary_mixing_rate_overview.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved overview figure:", fig_path)
plt.show()
